<a href="https://colab.research.google.com/github/Text-Machine/mask-predict/blob/main/example_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/> </a>

In [ ]:
#!git clone https://github.com/Text-Machine/mask-predict.git

In [ ]:
#%cd mask-predict

In [ ]:
#!pip install -e .

In [2]:
from explain import *
import pandas as pd
import json

In [ ]:
collection = 'hmd'
maskedToken = 'machine'
clusterName = 'slave'
modelName = "Livingwithmachines/bert_1760_1900"
dataPath = 'masking_data'
processedFolder = 'processed_results'

In this notebook we analyse the newspaper sentences where machine in mentioned by BLERT predicts words related to slavery.

In [13]:
#!unzip -o "{collection}_{maskedToken}_clusters.tsv.zip"

We rank sentences by their similarity to the 'slave' cluster based on the BLERT predictions. 

In [ ]:

data_df = pd.read_csv(f"{dataPath}/{collection}_{maskedToken}_clusters.tsv", sep="\t", index_col=0)
data_df = data_df.sort_values(by=f"{clusterName}_1760_1900", ascending=False).reset_index(drop=True)
data_df.head()

,item_code,issue_code,publication_code,prevSentence,currentSentence,markedSentence,maskedSentence,nextSentence,targetExpression,article_path,...,girl_1760_1900,slave_1760_1900,artisan_1760_1900,woman_1760_1900,machine_contemporary,boy_contemporary,girl_contemporary,slave_contemporary,artisan_contemporary,woman_contemporary
0,art0058,18540204,2645,He thinks the happiness and respectability of ...,Technical skill and exact work • mauship degra...,Technical skill and exact work • mauship degra...,Technical skill and exact work • mauship degra...,"It is not work that is objected to , but the n...",machine,0002645/1854/0204/0002645_18540204_art0058.txt,...,0.287,0.934,0.357,0.355,0.114,0.164,0.146,0.270,0.183,0.173
1,art0114,18650401,2194,The Secretary of War at Richmond has also auth...,Without this half the advantage of the measure...,Without this half the advantage of the measure...,Without this half the advantage of the measure...,The military news is still conflicting and unc...,machines,0002194/1865/0401/0002194_18650401_art0114.txt,...,0.301,0.797,0.374,0.360,0.176,0.201,0.214,0.322,0.222,0.249
2,art0016,18650403,2642,If we chance to turn back to the writings of t...,"Our fields seemed blasted, and no longer able ...","Our fields seemed blasted , and no longer able...","Our fields seemed blasted , and no longer able...",It was a time when the phrase Two Nations was ...,machines,0002642/1865/0403/0002642_18650403_art0016.txt,...,0.324,0.565,0.337,0.369,0.188,0.260,0.260,0.467,0.278,0.299
3,art0010,18641001,2194,Instead of each elector feeling elevated by th...,The act of receiving a bribe is of itself des ...,The act of receiving a bribe is of itself des ...,The act of receiving a bribe is of itself des ...,These and other similar reasons have so disgus...,machine,0002194/1864/1001/0002194_18641001_art0010.txt,...,0.257,0.563,0.283,0.298,0.142,0.190,0.194,0.268,0.143,0.205
4,art0098,18641001,2194,Instead of ech elector feeling elevated by the...,The act of receiving a bribe is of itself de •...,The act of receiving a bribe is of itself de •...,The act of receiving a bribe is of itself de •...,These and other similar reasons have so disgus...,machine,0002194/1864/1001/0002194_18641001_art0098.txt,...,0.269,0.558,0.293,0.309,0.123,0.177,0.180,0.256,0.136,0.195


We get the first 1000 sentences, and the top 5 predictions. This means we only look at context words where our target words of interest (like 'slave' appear in the top five words).

In [18]:
iloc_range = (0, 30)  # Adjust as needed
texts, predicted_targets = build_texts_targets(
    data_df, start=iloc_range[0], end=iloc_range[1],
    pred_col="pred_bert_1760_1900", top_n=5
)


We run the explainer using BLERT as the model we inspect, our targets are the top 5 predictions, the texts are the 1000 sentences selected in the previous cell.

In [ ]:
explainer = MaskedLMExplainer(model_name=modelName, device=pick_device())

Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie bert.embeddings.word_embeddings.weight to cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie cls.predictions.bias to cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BertForMaskedLM LOAD REPORT from: Livingwithmachines/bert_1760_1900
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
results = explainer.explain(texts, predicted_targets, word_agg="max")

In [ ]:
with open(f'results_{collection}_{maskedToken}_{clusterName}_pred.json', 'w') as f:
    json.dump(results, f)

Only retain results for context word contribution when slave or machine are predicted by BLERT.

In [ ]:
results_df = result_as_dataframe(results, ['slave','machine'])
results_df.to_csv(f'results_{collection}_{maskedToken}_{clusterName}_pred.csv')

Contrastive analysis, we use slave and machine as target words for all sentences, ignoring the actual predictions.

In [ ]:
manual_targets = [["slave","machine"] for _ in texts]
results2 = explainer.explain(texts, manual_targets, word_agg="max")


In [ ]:
with open(f'results_{collection}_{maskedToken}_{clusterName}_manual.json', 'w') as f:
    json.dump(results2, f)

In [ ]:
results_df = result_as_dataframe(results2, manual_targets[0])
results_df.to_csv(f'results_{collection}_{maskedToken}_{clusterName}_manual.csv')

# Load data

In [ ]:
df1 = pd.read_csv(f'{processedFolder}/results_manual.csv')
df2 = pd.read_csv(f'{processedFolder}/results_predicted.csv')
df1.shape, df2.shape

((23698, 5), (23698, 5))

In [ ]:

df_man = pd.read_csv(f'{processedFolder}/results_manual.csv', index_col=0)

In [9]:
df_man.head()

,Token,Score,Target,id
0,technical,0.037371,slave,0
1,skill,0.038192,slave,0
2,and,0.057331,slave,0
3,exact,0.002623,slave,0
4,work,0.106223,slave,0


In [25]:
slave_df = df_man[df_man['Target'] == 'slave'].groupby('Token').agg(avg=('Score', 'mean'), count=('Score', 'size'))
machine_df = df_man[df_man['Target'] == 'machine'].groupby('Token').agg(avg=('Score', 'mean'), count=('Score', 'size'))

In [28]:
machine_df.shape, slave_df.shape

((3394, 2), (387, 3))

In [26]:
slave_df['diff'] = slave_df['avg'] - machine_df['avg']

In [27]:
slave_df

,avg,count,diff
Token,,,
"""",0.032184,4,-0.031481
',0.010758,3,-0.038430
",",0.034779,77,-0.030880
-,0.021632,1,-0.004976
.,0.036517,31,0.016343
...,...,...,...
wronged,0.109848,1,NaN
yet,0.058362,1,0.048656
you,0.059283,1,0.012581


In [20]:
sentence = data_df.iloc[0]['maskedSentence']
highlight_context_tokens(explainer, sentence, target="slave", word_agg="max")

'\n    <div id="tokviz_8334111f25ad49a1a796376b5a729b03">\n      <div style=\'margin-bottom:6px;\'>\n        <b>Target:</b> <code>slave</code>\n      </div>\n      <div style=\'margin:6px 0 10px 0; font-size:13px; display:flex; gap:10px; align-items:center;\'>\n        <span style=\'background:rgba(30,136,229,0.35); padding:2px 8px; border-radius:4px;\'>&#9646; predicts</span>\n        <span style=\'background:rgba(229,57,53,0.35);  padding:2px 8px; border-radius:4px;\'>&#9646; opposes</span>\n        <span style=\'background:rgba(255,193,7,0.85);  padding:2px 8px; border-radius:4px; font-weight:bold;\'>[target] mask position</span>\n      </div>\n      <div style=\'line-height:2.4; font-size:15px;\'>\n        <span class=\'tok\' data-score=\'0.037372\' style=\'background:rgba(30, 136, 229, 0.155); padding:2px 4px; margin:1px; border-radius:4px; cursor:default;\'>technical</span> <span class=\'tok\' data-score=\'0.038193\' style=\'background:rgba(30, 136, 229, 0.156); padding:2px 4px; 